# 16. BatchNorm — стабильность и скорость обучения

Видео: [16. BatchNorm](https://www.youtube.com/watch?v=CSRu8byqxNs&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=17) (~4 мин)

## Зачем этот урок

Нормализация **входа** (видео 7 / `Normalize`) — на старте сети.  
**BatchNorm** — почти то же, но для **выходов слоёв**: следующий слой получает данные в удобном масштабе. Обучение обычно **быстрее и ровнее**.

> **Конспект:** [`d.4.3`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.3 Регуляризация в Deep Learning/d.4.3 Регуляризация в Deep Learning.md) **§2.2** (BN) + пример [`d.4.3.2`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.3 Регуляризация в Deep Learning/d.4.3.2 Пример — зачем BatchNorm.md). Карта — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).

**Не путать с Dropout:** Dropout — против зубрёжки; BN — **стабильность / скорость** (антипереобучение — слабый побочный эффект).

| блок | тема |
|:---|:---|
| 1 | `nn.BatchNorm1d` для Linear |
| 2 | `bias=False` у Linear + γ, β |
| 3 | `train` / `eval` (статистика батча vs running) |
| 4 | нюансы автора: не с Dropout на один слой; маленький batch |

**Как смотреть.** Целиком (~4 мин).

**Как решать.** Замени `...` → проверка. Решения — `16_batchnorm_решения.ipynb`.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


print("torch", torch.__version__)


## 1. <span style="color:#7CDBA0">`BatchNorm1d`</span> — для полносвязного слоя

Как у автора:
- к `Linear` → **`nn.BatchNorm1d(num_features)`**, `num_features` = число нейронов этого Linear (выход);
- к свёрткам позже → `BatchNorm2d` (каналы) — в следующих видео;
- ставить **до или после** активации — оба варианта ок, проверяют по задаче.

```python
nn.Linear(in_f, out_f, bias=False) → nn.BatchNorm1d(out_f) → nn.ReLU()
```


In [ ]:
# 1.1 слой BatchNorm1d
# аргумент — размер признака (число нейронов Linear, к которому цепляем BN)
bn = nn.BatchNorm1d(8)
print(bn)
need(isinstance(bn, nn.BatchNorm1d), "nn.BatchNorm1d")
need(bn.num_features == 8, "num_features = 8")
print("1.1 — верно")


### 1.2 Сеть с BN (имена слоёв — `Linear_`)

У автора: у Linear, к которому вешают BN, **`bias=False`** (смещение частично дублирует β у BN).


In [ ]:
# 1.2 TinyBN: Linear_1(bias=False) → BN → ReLU → Linear_2
class TinyBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.Linear_1 = nn.Linear(4, 8, bias=False)
        self.bn = nn.BatchNorm1d(8)
        self.act = nn.ReLU()
        self.Linear_2 = nn.Linear(8, 2)

    def forward(self, x):
        x = self.Linear_1(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.Linear_2(x)
        return x


torch.manual_seed(0)
net = TinyBN()
xb = torch.randn(16, 4) 
y = net(xb)
need(tuple(y.shape) == (16, 2), f"out {tuple(y.shape)}")
need(net.Linear_1.bias is None, "Linear_1: bias=False")
need(isinstance(net.bn, nn.BatchNorm1d) and net.bn.num_features == 8, "bn = BatchNorm1d(8)")
print("1.2 — верно", tuple(y.shape))


## 2. <span style="color:#7CDBA0">γ и β</span> — обучаемые

После нормировки к ~N(0,1) сеть может вернуть удобный масштаб:

`y = γ · x̂ + β`

В PyTorch это `bn.weight` (γ) и `bn.bias` (β) — они в `parameters()`.


In [ ]:
# 2.1 γ / β есть и требуют градиент
torch.manual_seed(0)
net = TinyBN()
need(net.bn.weight is not None and net.bn.bias is not None, "affine γ, β")
need(net.bn.weight.requires_grad and net.bn.bias.requires_grad, "γ, β обучаемые")
# имена в parameters
names = [n for n, _ in net.named_parameters()]
need(any("bn.weight" in n for n in names) and any("bn.bias" in n for n in names), "bn в parameters()")
print("params:", names)
print("2.1 — верно")


## 3. <span style="color:#7CDBA0">`train` / `eval`</span> — как у Dropout, но другая причина

| режим | BatchNorm |
|:---|:---|
| `train()` | μ, σ² **по текущему батчу** + обновляет running-статистику |
| `eval()` | берёт **накопленные** running_mean / running_var (батч из 1 картинки ок) |

Поэтому на val/test снова: `model.eval()` (+ обычно `torch.no_grad()`).


In [ ]:
# 3.1 один вход: в train два батча → разные выходы BN; в eval — стабильно
torch.manual_seed(0)
net = TinyBN()
# два разных батча одного размера
xb_a = torch.randn(32, 4)
xb_b = torch.randn(32, 4)

# train: прогон; eval: два одинаковых выхода; train≠eval на том же xb_a
ye1 = ...
ye2 = ...
yt = ...
need(torch.allclose(ye1, ye2), "в eval два одинаковых входа → одинаковый BN")
need(not torch.allclose(yt, ye1, atol=1e-5), "train (batch stats) ≠ eval (running) на том же xb")
print("3.1 — верно")


## 4. <span style="color:#7CDBA0">Нюансы автора</span>

1. **BN + Dropout на один слой** — лучше не вместе (ошибки не будет, толку мало).
2. **Маленький batch** (3–5) — BN шумный; от ~8 уже нормальнее.
3. Плюсы: ускоряет обучение; «против переобучения» — слабее, чем Dropout.

На собесе (из конспекта): *«Dropout — против переобучения; BatchNorm — стабильность и скорость сходимости»*.


In [ ]:
# 4.1 Fashion-подобная голова с BN (без Dropout)
class FashionLikeBN(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        # Flatten → Linear_1(..., bias=False) → BN → ReLU → Linear_2 → 10
        ...

    def forward(self, x):
        ...


torch.manual_seed(0)
m = FashionLikeBN()
xb = torch.randn(8, 1, 28, 28)
m.train()
yt = m(xb)
m.eval()
with torch.no_grad():
    ye = m(xb)
need(tuple(yt.shape) == (8, 10) and tuple(ye.shape) == (8, 10), "out (B, 10)")
need(m.Linear_1.bias is None, "bias=False у Linear_1")
need(any(isinstance(mod, nn.BatchNorm1d) for mod in m.modules()), "есть BatchNorm1d")
print("4.1 — верно")


## Итог

| факт | помнить |
|:---|:---|
| `BatchNorm1d(F)` | F = выход Linear |
| Linear + BN | часто `bias=False` |
| γ, β | обучаемые (`weight`, `bias` у BN) |
| `train` / `eval` | batch stats vs running |
| vs Dropout | BN ≠ «главное против зубрёжки» |
| вместе | BN+Dropout на один слой — лучше не надо |

Дальше — **видео 17**: свёрточные слои.  
Проект: раздел «Видео 16» в `project/fashion_mnist.ipynb`.
